In [1]:
import torch

/opt/homebrew/lib/python3.10/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import pandas as pd

class Dataset(torch.utils.data.Dataset):
  def __init__(self, csv) -> None:
      self.csv = csv
      self.data = pd.read_csv(csv)
      self.y = self.data['label'].values
      self.y = torch.tensor(self.y, dtype=torch.long)

      self.x = self.data.drop('label', axis=1).values.reshape(-1, 1, 28, 28)
      self.x = self.x / 255.0
      self.x = torch.tensor(self.x, dtype=torch.float32)

  def __len__(self):
      return len(self.data)
  def __getitem__(self, idx):
      return self.x[idx], self.y[idx]

In [4]:
train_dataset = Dataset('./data/train.csv')

In [7]:
train_dataset[5]


(tensor([[[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
           0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
           0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
           0.0000, 0.0000, 0.0000, 0.0000],
          [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
           0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
           0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
           0.0000, 0.0000, 0.0000, 0.0000],
          [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
           0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
           0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
           0.0000, 0.0000, 0.0000, 0.0000],
          [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
           0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
           0.0000, 0.0000, 0.0000, 0.0000, 

In [8]:
from torch import nn

class CNN(nn.Module):
  def __init__(self):
      super().__init__()
      self.conv1 = nn.Conv2d(1, 32, 3, 1)
      self.conv2 = nn.Conv2d(32, 64, 3, 1)
      self.dropout1 = nn.Dropout2d(0.25)
      self.dropout2 = nn.Dropout2d(0.5)
      self.fc1 = nn.Linear(9216, 128)
      self.fc2 = nn.Linear(128, 10)

  def forward(self, x):
      x = self.conv1(x)
      x = nn.functional.relu(x)
      x = self.conv2(x)
      x = nn.functional.relu(x)
      x = nn.functional.max_pool2d(x, 2)
      x = self.dropout1(x)
      x = torch.flatten(x, 1)
      x = self.fc1(x)
      x = nn.functional.relu(x)
      x = self.dropout2(x)
      x = self.fc2(x)
      output = nn.functional.log_softmax(x, dim=1)
      return output

In [10]:
train_dataset = Dataset('./data/train.csv')
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)


In [11]:
model = CNN()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.NLLLoss()

for epoch in range(10):
  for batch_idx, (inputs, target) in enumerate(train_loader):
    optimizer.zero_grad()
    output = model(inputs)
    loss = loss_fn(output, target)
    loss.backward()
    optimizer.step()

    if batch_idx % 100 == 0:
      print(f'Epoch: {epoch}, Batch: {batch_idx}, Loss: {loss}')
      

/opt/homebrew/lib/python3.10/site-packages/torch/nn/functional.py:1331: UserWarning: dropout2d: Received a 2-D input to dropout2d, which is deprecated and will result in an error in a future release. To retain the behavior and silence this warning, please use dropout instead. Note that dropout2d exists to provide channel-wise dropout on inputs with 2 spatial dimensions, a channel dimension, and an optional batch dimension (i.e. 3D or 4D inputs).
  warnings.warn(warn_msg)


Epoch: 0, Batch: 0, Loss: 2.3028793334960938
Epoch: 0, Batch: 100, Loss: 0.4621785879135132
Epoch: 0, Batch: 200, Loss: 0.48805832862854004
Epoch: 0, Batch: 300, Loss: 0.12474630028009415
Epoch: 0, Batch: 400, Loss: 0.05775921791791916
Epoch: 0, Batch: 500, Loss: 0.23654405772686005
Epoch: 0, Batch: 600, Loss: 0.11732658743858337
Epoch: 1, Batch: 0, Loss: 0.18851689994335175
Epoch: 1, Batch: 100, Loss: 0.23161323368549347
Epoch: 1, Batch: 200, Loss: 0.3437441885471344
Epoch: 1, Batch: 300, Loss: 0.09128528088331223
Epoch: 1, Batch: 400, Loss: 0.04437734931707382
Epoch: 1, Batch: 500, Loss: 0.11095494776964188
Epoch: 1, Batch: 600, Loss: 0.048853375017642975
Epoch: 2, Batch: 0, Loss: 0.055314015597105026
Epoch: 2, Batch: 100, Loss: 0.051304515451192856
Epoch: 2, Batch: 200, Loss: 0.04486243799328804
Epoch: 2, Batch: 300, Loss: 0.050422873347997665
Epoch: 2, Batch: 400, Loss: 0.053197719156742096
Epoch: 2, Batch: 500, Loss: 0.06005983054637909
Epoch: 2, Batch: 600, Loss: 0.02667332068085

In [12]:
class Learner:
  def __init__(self, epochs, model, optimizer, loss_fn, train_loader, valid_loader=None) -> None:
    self.epochs = epochs
    self.model = model
    self.optimizer = optimizer
    self.loss_fn = loss_fn
    self.train_loader = train_loader
    self.valid_loader = valid_loader

  def __call__(self):
    for epoch in range(self.epochs):
      for batch_idx, (inputs, target) in enumerate(train_loader):
        optimizer.zero_grad()
        output = model(inputs)
        loss = loss_fn(output, target)
        loss.backward()
        optimizer.step()
      
      print(f"epoch {epoch} loss {loss}")
      if self.valid_loader is not None:
        self.validate()
    print("Trained!")
  
  def validate(self):
    self.model.eval()
    with torch.no_grad():
      for _, (inputs, target) in enumerate(self.valid_loader):
        output = self.model(inputs)
        loss = self.loss_fn(output, target)
        print(f'Validation Loss: {loss}')

In [ ]:
train_dataset, valid_dataset = torch.utils.data.random_split(train_dataset, [0.8, 0.2])
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
valid_loader = torch.utils.data.DataLoader(valid_dataset, batch_size=64, shuffle=True)
learner = Learner(
  10, 
  model, 
  optimizer=torch.optim.Adam(model.parameters(), lr=0.001), 
  loss_fn=nn.NLLLoss(), 
  train_loader=train_loader, 
  valid_loader=valid_loader)